# ☀️ Solar Panel Mapping — Pont du Gard

Automated detection of solar photovoltaic installations across the  
**Communauté de Communes du Pont du Gard** (~230 km², southern France).

High-resolution IGN aerial imagery (20 cm/pixel, zoom 19) is fed tile by tile into a  
pre-trained **DeepLabV3 ResNet101** segmentation model. Each ~25 × 25 m tile receives a  
solar score; the results are aggregated into a territory-wide heatmap.

---

| | |
|:---|:---|
| **Model** | PV-Segmentation-deeplabv3 (Kleebauer et al., 2023) |
| **Performance** | F1: 95.27% — IoU: 91.04% |
| **Imagery** | IGN BD ORTHO — 20 cm/pixel |
| **Territory** | ~350,000 tiles at zoom 19 |
| **Course** | Data Science & AI — ESADE |

---

> ⚠️ **Always run Section 1 — Setup first at the start of every session.**  
> The model is loaded from Google Drive; no re-download is needed after the first run.

---
## 🔧 Section 1 — Setup

Run every cell in this section **once per Colab session** before anything else.

| Step | Cell | What it does |
|:---:|:---:|:---|
| 1 | 1.1 | Install Python packages |
| 2 | 1.2 | Import libraries |
| 3 | 1.3 | Detect GPU |
| 4 | 1.4 | Mount Google Drive and load the segmentation model |
| 5 | 1.5 | Define all helper functions |

In [ ]:
# Install all required packages.
# torch / torchvision : deep-learning model inference
# requests / pillow   : fetching and decoding IGN tile images
# tqdm                : progress bar for the tile loop
# matplotlib / pandas : visualisation and results handling
!pip install torch torchvision requests pillow tqdm matplotlib pandas -q
print('✅ Dependencies ready')

In [ ]:
# Standard-library and third-party imports.
import os, math, requests, torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torchvision import transforms   # Image pre-processing pipeline
from PIL import Image                # Image loading and resizing
from io import BytesIO               # In-memory buffer for HTTP responses
from tqdm import tqdm                # Progress bar
print('✅ Imports OK')

In [ ]:
# Detect available compute hardware.
# A GPU (T4 or better) is strongly recommended: CPU inference is ~10× slower
# and impractical for the full ~350,000-tile run.
# To enable: Runtime → Change runtime type → Hardware accelerator → T4 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ No GPU — enable it in Runtime > Change runtime type')

In [ ]:
# Mount Google Drive and load the segmentation model.
# The model weights (233 MB) live in your Drive at solar-project/.
# If the file is missing it is downloaded automatically from Fraunhofer
# ownCloud and saved to Drive for all future sessions.
from google.colab import drive
drive.mount('/content/drive')

MODEL_PATH = '/content/drive/MyDrive/solar-project/PV-Segmentation-deeplabv3.pt'

if not os.path.exists(MODEL_PATH):
    print('❌ Model not found — downloading from Fraunhofer...')
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    url = 'https://owncloud.fraunhofer.de/index.php/s/FdjVUCS8fgocAxz/download?path=%2F&files=PV-Segmentation-deeplabv3.pt'
    r = requests.get(url)
    with open(MODEL_PATH, 'wb') as f:
        f.write(r.content)
    print('✅ Model downloaded and saved to Drive')

# weights_only=False is required because the checkpoint stores the full model
# object (architecture + weights), not just a state dict.
model = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = model.to(device)
model.eval()   # Disable dropout / batch-norm training behaviour
print('✅ Model loaded from Drive on', device)

In [ ]:
# Global parameters and helper functions used throughout the pipeline.

# ── Parameters (validated through testing) ──────────────────────────────────
ZOOM      = 19     # Zoom 19 ≈ 25 × 25 m per tile — optimal for rooftop panels
IMG_SIZE  = 500    # Model input resolution; must be exactly 500 × 500 px
THRESHOLD = 0.5    # Pixel threshold: probabilities ≥ 0.5 are classified as solar

# Pre-processing applied to every tile before inference
TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()   # Converts PIL Image [0,255] → float tensor [0,1]
])


def lon_lat_to_tile(lon, lat, zoom):
    """Convert WGS-84 lon/lat to WMTS tile column and row (Web Mercator / TMS)."""
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    y = int((1 - math.log(math.tan(math.radians(lat)) + 1/math.cos(math.radians(lat))) / math.pi) / 2 * n)
    return x, y


def tile_to_lon_lat(col, row, zoom):
    """Convert tile column/row back to WGS-84 lon/lat (top-left corner of the tile)."""
    n = 2 ** zoom
    lon = col / n * 360 - 180
    lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * row / n))))
    return lon, lat


def get_ign_tile(col, row, zoom=ZOOM):
    """Fetch one IGN BD ORTHO aerial tile (20 cm/pixel) as a PIL RGB image.

    Returns None if the request times out or the server returns a non-image response.
    """
    url = (
        f'https://data.geopf.fr/wmts?SERVICE=WMTS&REQUEST=GetTile&VERSION=1.0.0'
        f'&LAYER=HR.ORTHOIMAGERY.ORTHOPHOTOS&STYLE=normal&TILEMATRIXSET=PM'
        f'&TILEMATRIX={zoom}&TILEROW={row}&TILECOL={col}&FORMAT=image/jpeg'
    )
    try:
        r = requests.get(url, timeout=15)
        if 'image' in r.headers.get('Content-Type', ''):
            return Image.open(BytesIO(r.content)).convert('RGB')
    except Exception:
        pass
    return None


def detect_solar(img, threshold=THRESHOLD):
    """Run the DeepLabV3 model on a PIL image.

    Returns:
        score (float) : percentage of pixels classified as solar panel (0–100)
        mask  (ndarray): binary H × W mask — 1 = solar panel, 0 = background
    """
    tensor = TRANSFORM(img).unsqueeze(0).to(device)   # Shape: [1, 3, 500, 500]
    with torch.no_grad():
        output = model(tensor)
    raw  = torch.sigmoid(output['out'].cpu()).squeeze().numpy()   # Probability map
    mask = (raw >= threshold).astype(int)
    score = mask.mean() * 100   # Fraction of solar pixels, expressed as a percentage
    return score, mask


print('✅ All functions ready — SETUP complete!')

---
## 🧪 Section 2 — Test Single Tile

Verify the full pipeline works end-to-end on a single tile **before** launching the large-scale run.

The test coordinates point to **Remoulins**, a town in the study area known to have  
visible rooftop solar installations.

| Expected outcome | |
|:---|:---|
| Tile fetches successfully | ✅ Image displays in cell 2.1 |
| Solar panels detected | ✅ Red overlay visible in cell 2.2 |
| Score > 0 | ✅ Printed solar score is non-zero |
| Tile fetch fails | ❌ Check network — re-run cell 2.1 |

In [ ]:
# Fetch a single IGN tile over Remoulins to confirm the WMTS API is reachable
# and returning valid aerial imagery.
LON_TEST, LAT_TEST = 4.5685, 43.9398   # Remoulins town centre
col, row = lon_lat_to_tile(LON_TEST, LAT_TEST, ZOOM)
print(f'Tile: zoom={ZOOM}, col={col}, row={row}')

img = get_ign_tile(col, row)
if img:
    print(f'✅ Tile fetched: {img.size}')
    plt.imshow(img)
    plt.title('IGN tile — Remoulins')
    plt.axis('off')
    plt.show()
else:
    print('❌ Failed to fetch tile — check your network and re-run this cell')

In [ ]:
# Run the segmentation model on the test tile and overlay the detection mask.
#
# Score interpretation:
#   0%        → no solar pixels detected
#   0.1–1%    → small residential panel(s)
#   1–5%      → significant rooftop installation
#   > 5%      → large rooftop or ground-mounted solar farm
if img is None:
    raise RuntimeError('❌ No image available — re-run cell 2.1 first')

score, mask = detect_solar(img)
print(f'Solar score: {score:.2f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(img)
ax1.set_title('IGN Image')
ax1.axis('off')
ax2.imshow(img.resize((IMG_SIZE, IMG_SIZE)))
ax2.imshow(mask, cmap='Reds', alpha=0.6)   # Red overlay = pixels classified as solar
ax2.set_title(f'Detection (score: {score:.1f}%)')
ax2.axis('off')
plt.tight_layout()
plt.show()

---
## 🗺️ Section 3 — Full Pipeline

Iterates over every tile in the bounding box, runs solar panel detection, and saves  
results to `solar-project/results.csv` on Drive after every column.  
The checkpoint system makes the run **safe to interrupt and resume** at any point.

| Mode | Tiles | Est. time (T4 GPU) |
|:---|:---:|:---:|
| `TEST_MODE = True` — Remoulins area | ~200 | < 5 min |
| `TEST_MODE = False` — Full territory | ~350,000 | Several hours |

> ⚠️ **GPU must be enabled** before starting the full run.  
> Results accumulate in `solar-project/results.csv` on your Drive.

In [ ]:
# Define the geographic bounding box and compute the corresponding tile range.
#
# Set TEST_MODE = True  for a quick validation run (~200 tiles, < 5 min)
#     TEST_MODE = False for the full territory      (~350,000 tiles)
TEST_MODE = True

if TEST_MODE:
    BBOX = {'lon_min': 4.555, 'lon_max': 4.585, 'lat_min': 43.930, 'lat_max': 43.950}
else:
    BBOX = {'lon_min': 4.40, 'lon_max': 4.65, 'lat_min': 43.88, 'lat_max': 44.02}

# Note: in Web Mercator tile indexing, row increases southward.
# Therefore the southern edge (lat_min) maps to the highest row index (row_max).
col_min, row_max = lon_lat_to_tile(BBOX['lon_min'], BBOX['lat_min'], ZOOM)
col_max, row_min = lon_lat_to_tile(BBOX['lon_max'], BBOX['lat_max'], ZOOM)

total = (col_max - col_min) * (row_max - row_min)
print(f'Bounding box: {BBOX}')
print(f'Tile range: cols {col_min}→{col_max}, rows {row_min}→{row_max}')
print(f'Total tiles: ~{total:,}')
print(f'Estimated time on GPU (1 tile/s): ~{total/60:.0f} minutes')

In [ ]:
# Main detection loop with checkpoint support.
# Results are written to Drive after each column so the run can be interrupted
# and resumed without reprocessing tiles that are already in results.csv.
RESULTS_PATH = '/content/drive/MyDrive/solar-project/results.csv'

# Resume from an existing results file if one exists
if os.path.exists(RESULTS_PATH):
    df_existing = pd.read_csv(RESULTS_PATH)
    done = set(zip(df_existing['col'].astype(int), df_existing['row'].astype(int)))
    print(f'Resuming — {len(done)} tiles already processed')
else:
    df_existing = pd.DataFrame()
    done = set()
    print('Starting fresh')

results = []
errors  = 0

for c in tqdm(range(col_min, col_max), desc='Processing columns'):
    for r in range(row_min, row_max):
        if (c, r) in done:
            continue   # Skip tiles already processed in a previous run
        try:
            img_tile = get_ign_tile(c, r)
            if img_tile is None:
                continue   # IGN API returned no image for this tile
            score, _ = detect_solar(img_tile)
            lon, lat = tile_to_lon_lat(c, r, ZOOM)
            results.append({'col': c, 'row': r, 'lon': lon, 'lat': lat, 'score': score})
        except Exception:
            errors += 1
            continue

    # Checkpoint: flush this column's results to Drive before moving on
    if results:
        df_new = pd.DataFrame(results)
        df_all = pd.concat([df_existing, df_new], ignore_index=True)
        df_all.to_csv(RESULTS_PATH, index=False)
        df_existing = df_all
        done.update([(int(entry['col']), int(entry['row'])) for entry in results])
        results = []

print(f'\n✅ Done! Total processed: {len(df_existing):,} tiles, Errors: {errors}')

---
## 📊 Section 4 — Results & Heatmap

Load the accumulated `results.csv` and generate the final solar potential map.

| Cell | Output |
|:---:|:---|
| 4.1 | Summary statistics + score distribution histogram |
| 4.2 | Territory-wide heatmap saved to `solar-project/heatmap.png` on Drive |
| 4.3 | Top 20 highest-scoring tiles |

> Run Section 3 first — these cells require `results.csv` to exist on Drive.

In [ ]:
# Load results and print summary statistics.
# 'score' is the percentage of pixels classified as solar panel (0–100).
# A threshold of 0.5% filters background noise while retaining real detections.
RESULTS_PATH = '/content/drive/MyDrive/solar-project/results.csv'
df = pd.read_csv(RESULTS_PATH)

print(f'Total tiles processed : {len(df):,}')
print(f'Tiles with panels (score > 0.5%) : {(df.score > 0.5).sum():,}')
print(f'Average score : {df.score.mean():.3f}%')
print(f'Max score : {df.score.max():.2f}%')

# Score distribution — most tiles cluster near 0; rightward peaks indicate solar zones
df['score'].hist(bins=50, figsize=(10, 4))
plt.title('Distribution of solar scores')
plt.xlabel('Score (%)')
plt.ylabel('Number of tiles')
plt.show()

In [ ]:
# Generate the territory-wide solar heatmap.
# Each dot represents one ~25 × 25 m tile; colour encodes solar score:
#   green → low / no solar   |   red → high solar density
# The colour scale is capped at the 95th-percentile score so that a few
# extreme outlier tiles do not wash out the rest of the palette.
df = pd.read_csv(RESULTS_PATH)

fig, ax = plt.subplots(figsize=(14, 10))
sc = ax.scatter(
    df['lon'], df['lat'],
    c=df['score'],
    cmap='RdYlGn_r',
    s=3,
    alpha=0.8,
    vmin=0,
    vmax=df['score'].quantile(0.95)   # Cap at 95th percentile for better contrast
)
plt.colorbar(sc, ax=ax, label='Solar panel density score (%)')
ax.set_title('Solar Panel Mapping — Communauté de Communes du Pont du Gard', fontsize=14)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()

HEATMAP_PATH = '/content/drive/MyDrive/solar-project/heatmap.png'
plt.savefig(HEATMAP_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Heatmap saved to {HEATMAP_PATH}')

In [ ]:
# List the 20 tiles with the highest solar scores.
# The 1.0% threshold filters out background noise and marginal detections,
# retaining only tiles with a clearly visible solar installation.
# These are the best candidates for field verification or planning analysis.
df = pd.read_csv(RESULTS_PATH)
top_zones = df[df['score'] > 1.0].nlargest(20, 'score')
print('Top 20 tiles with highest solar density:')
print(top_zones[['lon', 'lat', 'score']].to_string(index=False))